# Chapter 4 — Parser Combinators

Before we write more Bluespec, we are going to build something concrete in plain
Haskell: a parser. Not because parsers are the point, but because building one
will force us to confront a problem — and the solution to that problem is the
same pattern Bluespec uses everywhere.

We will get to that pattern in Chapter 5. For now, we build the parser the slow
way, so that the need for something better becomes undeniable.

## What is a parser?

A parser reads input from left to right and tries to recognize structure in it.
At each step it has two things:

- What it has recognized so far
- What input it hasn't touched yet

Consider parsing the string `"123+45"` as a number. A number parser chews
through `"123"`, recognizes the integer `123`, and hands back `"+45"` as the
unconsumed remainder. That remainder is then available for the *next* parser.
This is how parsers chain: each one takes what it needs and leaves the rest.

### Wrong turns and backtracking

Parsers don't always know in advance which branch applies. If you try a number
parser and the input starts with `'('`, it fails. You need to *backtrack* —
pretend the attempt never happened and try something else on the original input.

In traditional compilers, backtracking uses an explicit stack: push your position
before trying a branch, pop it on failure. In our combinator parser, we get
backtracking for free: we never mutate the input string, we just pass it down as
a function argument. A failed branch returns a failure value; the original string
is unchanged and available to try again.

## The types

We need two types. First, the result of a parse attempt:

In [1]:
data ParseResult a
    = NoParse
    | ParseOk { consumed :: a, remaining :: String }
    deriving Show

`ParseResult a` has two constructors:

- `NoParse` — the parse failed, nothing was recognized
- `ParseOk` — success, with two named fields:
  - `consumed` — the structure that was recognized (type `a`)
  - `remaining` — the input that wasn't touched, ready for the next parser

Think of a successful `ParseOk` as the parser's outgoing message:
*"here's what I understood, here's what I left for you."*

Now the parser itself:

In [2]:
data Parser a = Parser { runParser :: String -> ParseResult a }

Line 1: Use newtype instead of data
Found:
data Parser a = Parser {runParser :: String -> ParseResult a}
Why not:
newtype Parser a = Parser {runParser :: String -> ParseResult a}

`Parser a` wraps a single function: given the current input string, produce a
`ParseResult a`.

The `data` wrapper is bookkeeping — it lets us give this type a name and attach
behaviour to it later. `runParser` is a record field, which means it doubles as
a function that *extracts* the wrapped function from a `Parser`. When you write
`runParser p`, you are pulling the function out of the `Parser` wrapper `p` so
you can call it.

> **Aside:** You will often see `newtype` instead of `data` for single-field
> wrappers like this. `newtype` is identical in behaviour but carries a compiler
> guarantee that the wrapper costs nothing at runtime. We use `data` here because
> you have already seen it. Chapter 6 will revisit this.

## `item` — the only true primitive

`item` is the one parser that directly touches the input string. Everything else
will be built on top of it.

In [3]:
item :: Parser Char
item = Parser itemFunction
  where
    itemFunction []     = NoParse
    itemFunction (c:cs) = ParseOk { consumed = c, remaining = cs }

We define a named helper function `itemFunction` and pass it to the `Parser`
constructor — no anonymous functions yet.

`itemFunction` pattern-matches on the input string:
- `[]` — empty input, nothing to consume, return `NoParse`
- `(c:cs)` — Haskell's list pattern simultaneously names the first character `c`
  and the rest of the string `cs`. We return `ParseOk` with `c` consumed and
  `cs` remaining.

Let's try it:

In [4]:
runParser item "hello"

ParseOk {consumed = 'h', remaining = "ello"}

In [5]:
runParser item ""

NoParse

## Building bigger parsers from small ones

This is the key idea of parser *combinators*: we build larger parsers by
composing smaller ones. Each new parser is a function that takes simpler parsers
as building blocks. We are doing function composition, just with parsers as the
values being composed.

### `satisfy` — parsing with a condition

Sometimes we want to consume a character only if it meets some condition — is it
a digit? A letter? A specific symbol? `satisfy` takes a test function and builds
a parser that only succeeds when the test passes.

In [6]:
satisfy :: (Char -> Bool) -> Parser Char
satisfy test = Parser satisfyFunction
  where
    satisfyFunction input =
        case runParser item input of
            NoParse                                    -> NoParse
            ParseOk { consumed = c, remaining = rest } ->
                if test c
                then ParseOk { consumed = c, remaining = rest }
                else NoParse

`satisfy` runs `item` to get a character, then checks the test:
- If `item` returned `NoParse`, propagate failure
- If `item` returned `ParseOk`, check `f c`:
  - Test passes: return the same `ParseOk`
  - Test fails: return `NoParse`

Notice `runParser item input` — we are extracting the function from `item` using
the `runParser` field accessor, then applying it to `input`.

### `char`, `digit`, `natural`

Each one built directly from the last:

In [7]:
import Data.Char (isDigit)

In [8]:
char :: Char -> Parser Char
char target = satisfy isTarget
  where
    isTarget c = c == target

digit :: Parser Char
digit = satisfy isDigit

In [9]:
runParser (char 'h') "hello"

ParseOk {consumed = 'h', remaining = "ello"}

In [10]:
runParser (char 'z') "hello"

NoParse

In [11]:
runParser digit "42abc"

ParseOk {consumed = '4', remaining = "2abc"}

## `<|>` — choice

We need a way to say "try this parser, and if it fails, try that one instead."
We define the `<|>` operator for this:

In [12]:
(<|>) :: Parser a -> Parser a -> Parser a
(<|>) p q = Parser choiceFunction
  where
    choiceFunction input =
        case runParser p input of
            NoParse -> runParser q input
            result  -> result

infixl 3 <|>

Try `left` on the input. If it returns `NoParse`, try `right` on the *same*
original input — this is the backtracking: the input was never mutated, so
retrying costs nothing. If `left` succeeds, return that result immediately.

## `many` and `some` — repetition

We need to parse sequences of things. `some` parses one or more occurrences;
`many` parses zero or more.

In [13]:
many :: Parser a -> Parser [a]
many p = Parser manyFunction
  where
    manyFunction input =
        case runParser (some p) input of
            NoParse  -> ParseOk { consumed = [], remaining = input }
            result   -> result

some :: Parser a -> Parser [a]
some p = Parser someFunction
  where
    someFunction input =
        case runParser p input of
            NoParse                                    -> NoParse
            ParseOk { consumed = x, remaining = rest } ->
                case runParser (many p) rest of
                    NoParse                                      -> NoParse
                    ParseOk { consumed = xs, remaining = rest2 } ->
                        ParseOk { consumed = x : xs, remaining = rest2 }

> **Warning:** `many p` where `p` never fails will loop forever. `many` has no
> way to detect that it isn't making progress.

Now we can build `natural` — a parser for multi-digit integers:

In [14]:
natural :: Parser Int
natural = Parser naturalFunction
  where
    naturalFunction input =
        case runParser (some digit) input of
            NoParse                                     -> NoParse
            ParseOk { consumed = ds, remaining = rest } ->
                ParseOk { consumed = read ds, remaining = rest }

In [15]:
runParser natural "123abc"

ParseOk {consumed = 123, remaining = "abc"}

In [16]:
runParser natural "abc"

NoParse

## The Calculator

We now have enough to build a calculator parser. Here is the grammar:

```
expr   ::= term   (('+' | '-') term)*
term   ::= factor (('*' | '/') factor)*
factor ::= natural | '(' expr ')'
```

The layered structure gives us operator precedence for free: `term` is parsed as
a unit before `expr` ever sees it, so `*` binds tighter than `+`. Parentheses
re-enter at `expr` level, resetting the hierarchy. `factor` calling `expr`
makes the whole thing recursive.

We need operator parsers that return functions, not just characters:

In [17]:
-- Thread two parsers: run the first, pass its consumed value to a function
-- that produces the second parser, run that on the remaining input.
andThen :: Parser a -> (a -> Parser b) -> Parser b
andThen p f = Parser andThenFunction
  where
    andThenFunction input =
        case runParser p input of
            NoParse                                    -> NoParse
            ParseOk { consumed = x, remaining = rest } -> runParser (f x) rest

-- Yield a value without consuming any input.
yield :: b -> Parser b
yield val = Parser yieldFunction
  where
    yieldFunction input = ParseOk { consumed = val, remaining = input }

charToAddOp :: Char -> Parser (Int -> Int -> Int)
charToAddOp '+' = yield (+)
charToAddOp _   = yield (-)

charToMulOp :: Char -> Parser (Int -> Int -> Int)
charToMulOp '*' = yield (*)
charToMulOp _   = yield div

addOp :: Parser (Int -> Int -> Int)
addOp = andThen (char '+' <|> char '-') charToAddOp

mulOp :: Parser (Int -> Int -> Int)
mulOp = andThen (char '*' <|> char '/') charToMulOp

Notice `andThen` and `yield` — we needed to sequence two parsers and thread\n",
    "`remaining` from one into the next, so we wrote that logic as a named\n",
    "combinator. `yield` produces a value without consuming any input. We will\n",
    "keep needing both of these. Keep that in mind.\n",
    "\n",
    "Now the grammar itself. Notice how `remaining` is explicitly passed at every step:"

In [18]:
chain :: Parser (Int -> Int -> Int) -> Parser Int -> Int -> Parser Int
chain opParser operandParser acc = Parser chainFunction
  where
    chainFunction input =
        case runParser opParser input of
            NoParse ->
                ParseOk { consumed = acc, remaining = input }
            ParseOk { consumed = f, remaining = rest1 } ->
                case runParser operandParser rest1 of
                    NoParse ->
                        ParseOk { consumed = acc, remaining = input }
                    ParseOk { consumed = x, remaining = rest2 } ->
                        runParser (chain opParser operandParser (f acc x)) rest2

factor :: Parser Int
factor = Parser factorFunction
  where
    factorFunction input =
        case runParser natural input of
            ParseOk { consumed = n, remaining = rest } ->
                ParseOk { consumed = n, remaining = rest }
            NoParse ->
                case runParser (char '(') input of
                    NoParse -> NoParse
                    ParseOk { consumed = _, remaining = rest1 } ->
                        case runParser expr rest1 of
                            NoParse -> NoParse
                            ParseOk { consumed = n, remaining = rest2 } ->
                                case runParser (char ')') rest2 of
                                    NoParse -> NoParse
                                    ParseOk { consumed = _, remaining = rest3 } ->
                                        ParseOk { consumed = n, remaining = rest3 }

term :: Parser Int
term = Parser termFunction
  where
    termFunction input =
        case runParser factor input of
            NoParse                                    -> NoParse
            ParseOk { consumed = n, remaining = rest } ->
                runParser (chain mulOp factor n) rest

expr :: Parser Int
expr = Parser exprFunction
  where
    exprFunction input =
        case runParser term input of
            NoParse                                    -> NoParse
            ParseOk { consumed = n, remaining = rest } ->
                runParser (chain addOp term n) rest

In [19]:
runParser expr "1+2*3"

ParseOk {consumed = 7, remaining = ""}

In [20]:
runParser expr "(1+2)*3"

ParseOk {consumed = 9, remaining = ""}

In [21]:
runParser expr "2*(3+4*(5-1))"

ParseOk {consumed = 38, remaining = ""}

It works. But look at `factor`. We wrote `remaining` eight times in one function.
Every step of the parse requires a `case` block, a name for the result, and an
explicit handoff of `remaining` to the next step. And `andThen` — we wrote that
same threading logic twice, once for `addOp` and once for `mulOp`.

This is not a Haskell problem. This is not even a bad solution. It is the
unavoidable noise of manually threading state through every step of a
computation. Every time we chain two parsers, we repeat the same pattern:
run a parser, check for failure, extract `remaining`, pass it on.

In Chapter 5, we will name that pattern and encode it once.